In [1]:
import os
import ast
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 1000)

from tqdm import tqdm

from openai import AzureOpenAI

pd.set_option('display.max_columns', 100)

### Load OpenAI Model

In [2]:
os.environ["AZURE_OPENAI_KEY"] = ""
os.environ["AZURE_OPENAI_ENDPOINT"] = ""
os.environ["AZURE_API_VERSION"] = ""
os.environ["AZURE_DEPLOYMENT_ID"] = ""
os.environ["AWS_ACCESS_KEY"] = ""
os.environ["AWS_SECRET_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"] = ""
model_name = "gpt-4o-mini"

client = AzureOpenAI(
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_API_VERSION"),
    azure_deployment=os.getenv("AZURE_DEPLOYMENT_ID")
)

### Taxonomy Functions

In [3]:
def clean_and_parse(x):
    import ast 

    if isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        cleaned = x.replace('""', '"')
        cleaned = cleaned.replace('\n', ' ').replace('\r', '')  # Remove line breaks, if any

        try:
            return ast.literal_eval(cleaned)
        except Exception as e:
            print(f"Error parsing string: {cleaned}\n{e}")
            return x
    else:
        return x

def get_main_taxonomy_examples(dom: str, mode: str) -> str:

    if mode == "CC":
    
        df = pd.read_csv("../semeval-task-10/cc_taxonomy.csv")
        df = df.astype(str)
        df['Main Narrative Example'] = df['Main Narrative Example'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x)
                
        taxonomy = ''
        examples = ''

        dom = dom.split(":")[0] if len(dom.split(":")) > 0 else dom

        if dom in df['Main Narrative'].values:
            temp = df[df['Main Narrative'] == dom].reset_index()         
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Main Narrative']} | Definition: {temp.loc[0, 'Main Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Main Narrative Example'], list):
                for item in temp.loc[0, 'Main Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Main Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Main Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Main Narrative']}\n"
            else:
                pass


        return (taxonomy, examples)

    else:

        df = pd.read_csv("../semeval-task-10/urw_taxonomy.csv")
        df = df.astype(str)
        df['Main Narrative Example'] = df['Main Narrative Example'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') and x.endswith(']') else x)
          
        taxonomy = ''
        examples = ''

        dom = dom.split(":")[0] if len(dom.split(":")) > 0 else dom

        if dom in df['Main Narrative'].values:
            temp = df[df['Main Narrative'] == dom].reset_index()           
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Main Narrative']} | Definition: {temp.loc[0, 'Main Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Main Narrative Example'], list):
                for item in temp.loc[0, 'Main Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Main Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Main Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Main Narrative']}\n"
            else:
                pass
            
        return (taxonomy, examples)

def get_sub_taxonomy_examples(sub: str, mode: str) -> str:

    if mode == "CC":
        df = pd.read_csv("../semeval-task-10/cc_taxonomy.csv")
        df = df.astype(str)
        df['Sub Narrative Example'] = df['Sub Narrative Example'].apply(clean_and_parse)

        taxonomy = ''
        examples = ''

        sub = sub.split(":")[2][1:] if len(sub.split(":")) > 1 else sub

        if sub in df['Sub Narrative'].values:
            temp = df[df['Sub Narrative'] == sub].reset_index()
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Sub Narrative']} | Definition: {temp.loc[0, 'Sub Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Sub Narrative Example'], list):
                for item in temp.loc[0, 'Sub Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Sub Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Sub Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Sub Narrative']}\n"
            else:
                pass
        return (taxonomy, examples)

    else:
        df = pd.read_csv("../semeval-task-10/urw_taxonomy.csv")
        df = df.astype(str)
        df['Sub Narrative Example'] = df['Sub Narrative Example'].apply(clean_and_parse)

        taxonomy = ''
        examples = ''

        sub = sub.split(":")[2][1:] if len(sub.split(":")) > 1 else sub

        if sub in df['Sub Narrative'].values:
            temp = df[df['Sub Narrative'] == sub].reset_index()
            taxonomy = taxonomy + f"Category: {temp.loc[0, 'Sub Narrative']} | Definition: {temp.loc[0, 'Sub Narrative Definition']}\n"
            if isinstance(temp.loc[0, 'Sub Narrative Example'], list):
                for item in temp.loc[0, 'Sub Narrative Example']:
                    examples = examples + f"{item} => {temp.loc[0, 'Sub Narrative']}\n"
                if not temp.loc[0, 'Detail Instructions Sub Narrative'] == 'nan':
                    examples = examples + f"Note: {temp.loc[0, 'Detail Instructions Sub Narrative']}\n"
            else:
                pass
        return (taxonomy, examples)

In [4]:
def create_prompt(text, dom, sub, mode):
    # Helper function replacing quotation marks in the text:
    replace_qm = lambda s: s.replace('"', "'")

    language = 'BULGARIAN'

    if mode == "CC":
        # Update predicted_labels by slicing from the 4th character
        main_taxonomy, main_examples = get_main_taxonomy_examples(dom, mode)
        sub_taxonomy, sub_examples = get_sub_taxonomy_examples(sub, mode)
        context = f"""You will be given an article along with the dominant narrative and sub narrative associated with the article. 
        
        GOAL: Justify the choice of dominant and sub narratives assigned to the article in {language}. Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.

            INSTRUCTIONS:
            Read the provided text carefully.
            You will be given: 
            1.Taxonomy - definition of the narrative
            2.List of relevant examples - sentences which determine which justify the narrative and align with its defintion 
            3.Any additional idenyifying information for the narratives.
            Based on the given information give an explanation as to why the dominant and sub narratives are the correct choice for the text file.
            
            Note: Keep the output concise and to the point - ideally find relevant textual examples.

            DOMINANT NARRATIVE: {dom[4:]}
            TAXONOMY: {main_taxonomy}
            RELEVANT EXAMPLES: 
            {main_examples}

            SUB NARRATIVE: {sub[4:]}
            TAXONOMY: {sub_taxonomy}
            RELEVANT EXAMPLES:
            {sub_examples}
        """

        prompt = f'''{context}
        -------------------------------------------------------
        Based on the given Instructions, Taxonomies and Examples: Justify the choice of dominant and sub narratives assigned to the Climate Change article in {language} within 80 words.

        ARTICLE TEXT TO PREDICT: "{replace_qm(text)}" => '''
        
        return {
            "role": "user",
            "content": prompt
        }
        
    else:
        # Update predicted_labels by slicing from the 5th character
        main_taxonomy, main_examples = get_main_taxonomy_examples(dom, mode)
        sub_taxonomy, sub_examples = get_sub_taxonomy_examples(sub, mode)

        context = f"""You will be given an article along with the dominant narrative and sub narrative associated with the article. 
        
        GOAL: Justify the choice of dominant and sub narratives assigned to the article in {language}. Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.

            INSTRUCTIONS:
            Read the provided text carefully.
            You will be given: 
            1.Taxonomy - definition of the narrative
            2.List of relevant examples - sentences which determine which justify the narrative and align with its defintion 
            3.Any additional idenyifying information for the narratives.
            Based on the given information give an explanation as to why the dominant and sub narratives are the correct choice for the text file.
            
            Note: Keep the output concise and to the point - ideally find relevant textual examples.

            DOMINANT NARRATIVE: {dom[4:]}
            TAXONOMY: {main_taxonomy}
            RELEVANT EXAMPLES: 
            {main_examples}

            SUB NARRATIVE: {sub[4:]}
            TAXONOMY: {sub_taxonomy}
            RELEVANT EXAMPLES:
            {sub_examples}
        """

        prompt = f'''{context}
        -------------------------------------------------------
        Based on the given Instructions, Taxonomies and Examples: Justify the choice of dominant and sub narratives assigned to the Climate Change article in {language} within 80 words.

        ARTICLE TEXT TO PREDICT: "{replace_qm(text)}" => '''
        
        return {
            "role": "user",
            "content": prompt
        }

In [5]:
def get_embedded_json(embedded_str):
    import re
    try:
        # Extract only the list content using regex
        match = re.search(r"\[.*\]", embedded_str)
        if not match:
            return []  # Return empty list if no valid list is found
        
        cleaned_str = match.group(0)  # Extract the matched list portion

        # Convert to Python list safely
        return ast.literal_eval(cleaned_str)
    
    except (ValueError, SyntaxError):
        return []  # Return empty list if parsing fails

In [11]:
def generate_response(text:str, dom, sub, mode, temp):

    language = 'BULGARIAN'

    system_main = \
'''You are an expert trained to analyse and justify the choice of dominant and sub narratives assigned to a given article within 80 words.

Instructions:

Write in {language} only.
Use ReACT (Reasoning and Contextual Text) to justify the choice of dominant and sub narratives assigned to the article.
Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.
Keep the output concise and to the point—ideally find relevant textual examples.

Categorization Rules:

Use the help of the provided taxonomy and examples to justify the choice of dominant and sub narratives assigned to the article.

OUTPUT FORMAT:
Return {language} language text in paragraph(s) format within 80 words.
'''

    
    prompt = create_prompt(text, dom, sub, mode)

    messages = [
        {"role": "system", "content": system_main},
        {"role": "user", "content": prompt.get("content", "")}
    ]
    
    response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=temp,
    max_tokens=100
    )   

    output = str(response.choices[0].message.content)
    return output



def generate_response_with_retries(text: str, dom, sub, mode, temp, retries=3, backoff_factor=0.5):
    language = 'BULGARIAN'

    system_main = f'''
You are an expert trained to analyse and justify the choice of dominant and sub narratives assigned to a given article within 80 words.

Instructions:
- Write in {language} only.
- Use ReACT (Reasoning and Contextual Text) to justify the choice of dominant and sub narratives assigned to the article.
- Provide reasoning and quote relevant text from the original text as to why these are the correct choice of dominant and sub narratives for the text file.
- Keep the output concise and to the point—ideally find relevant textual examples.

Categorization Rules:
- Use the help of the provided taxonomy and examples to justify the choice of dominant and sub narratives assigned to the article.

OUTPUT FORMAT:
- Return {language} language text in paragraph(s) format within 80 words.
'''

    prompt = create_prompt(text, dom, sub, mode)
    messages = [
        {"role": "system", "content": system_main},
        {"role": "user", "content": prompt.get("content", "")}
    ]

    # Retry mechanism with exponential backoff
    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=temp,
                max_tokens=100
            )
            output = str(response.choices[0].message.content)
            return output  # Return the response if successful

        except (openai.error.Timeout, openai.error.APIError, openai.error.RateLimitError) as e:
            print(f"Attempt {attempt} failed with error: {e}")
            
            if attempt < retries:
                # Exponential backoff
                sleep_time = backoff_factor * (2 ** (attempt - 1))
                print(f"Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)
            else:
                print(f"Failed after {retries} attempts.")
                return f"Error: {e}"

        except Exception as e:
            # Catch unexpected errors
            print(f"Unexpected error: {e}")
            return f"Unexpected Error: {e}"


### Importing the data

In [12]:
df = pd.read_csv('SemEval 2025 Test Data/final_datasets/bulgarian_gold_label_data.csv')

In [13]:
df.shape

(357, 5)

In [14]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text
0,BG_670.txt,URW: Blaming the war on others rather than the...,URW: Blaming the war on others rather than the...,"Обвинява се управлението на Запада,че е претър...",Опитът на колективния Запад да „обезкърви Руси...
1,BG_3245.txt,"URW: Discrediting the West, Diplomacy","URW: Discrediting the West, Diplomacy: The Wes...",Коментира се евентуално прекратяване на подкре...,Подкрепата за Киев от страна на Запада вече не...
2,A9_BG_5190.txt,"URW: Discrediting the West, Diplomacy",none,Неправителствената организация - Международнат...,"Дмитрий Медведев: НПО-та, спонсорирани от Соро..."
3,A9_BG_3379.txt,"URW: Discrediting the West, Diplomacy",none,Санкциите и политиката на ЕС и САЩ след 2014-т...,Британски дипломат обвини Запада за украинския...
4,A7_URW_BG_3566.txt,URW: Discrediting Ukraine,URW: Discrediting Ukraine: Ukraine is a puppet...,В статията на няколко пъти САЩ са обвинени в т...,Ответните мерки ще бъдат крайно болезнени за Е...


In [15]:
df['mode'] = df['dominant_narrative'].apply(lambda x: 'CC' if x.split(':')[0] == 'CC' else 'URW')

In [16]:
df['mode'].value_counts()

mode
URW    250
CC     107
Name: count, dtype: int64

In [17]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text,mode
0,BG_670.txt,URW: Blaming the war on others rather than the...,URW: Blaming the war on others rather than the...,"Обвинява се управлението на Запада,че е претър...",Опитът на колективния Запад да „обезкърви Руси...,URW
1,BG_3245.txt,"URW: Discrediting the West, Diplomacy","URW: Discrediting the West, Diplomacy: The Wes...",Коментира се евентуално прекратяване на подкре...,Подкрепата за Киев от страна на Запада вече не...,URW
2,A9_BG_5190.txt,"URW: Discrediting the West, Diplomacy",none,Неправителствената организация - Международнат...,"Дмитрий Медведев: НПО-та, спонсорирани от Соро...",URW
3,A9_BG_3379.txt,"URW: Discrediting the West, Diplomacy",none,Санкциите и политиката на ЕС и САЩ след 2014-т...,Британски дипломат обвини Запада за украинския...,URW
4,A7_URW_BG_3566.txt,URW: Discrediting Ukraine,URW: Discrediting Ukraine: Ukraine is a puppet...,В статията на няколко пъти САЩ са обвинени в т...,Ответните мерки ще бъдат крайно болезнени за Е...,URW


In [18]:
# tqdm.pandas()

# for i in np.arange(0.1, 1, 0.1):
#     print("Iterating at temperature: ", i.round(2))
#     temp = i.round(2)
#     df[f'output_temp_{temp}'] = df.progress_apply(lambda x: generate_response(x['text'], x['dominant_narrative'], x['sub_narratives'], x['mode'], temp), axis=1)

# Enable tqdm progress bar for parallel processing
from concurrent.futures import ThreadPoolExecutor, as_completed
tqdm.pandas()

# Function to handle each API call
def process_row(row, temp):
    return generate_response_with_retries(row['text'], row['dominant_narrative'], row['sub_narratives'], row['mode'], temp)


# Function to process each temperature
def process_temperature(temp, df):
    temp = round(temp, 2)
    tqdm.write(f"Iterating at temperature: {temp}")

    # Use ThreadPoolExecutor for concurrent API calls
    with ThreadPoolExecutor(max_workers=25) as executor:
        futures = {executor.submit(process_row, row, temp): idx for idx, row in df.iterrows()}

        # Collect results with progress tracking
        results = []
        for future in tqdm(as_completed(futures), total=len(futures), desc=f'Temp {temp}'):
            idx = futures[future]
            try:
                result = future.result()
            except Exception as e:
                result = f"Error: {e}"  # Handle API errors gracefully
            results.append((idx, result))

    # Assign results back to DataFrame
    output_column = f'output_temp_{temp}'
    df[output_column] = pd.Series(dict(results))

    return df[[output_column]]

# Parallel temperature processing
final_results = []
for temp in [0.2, 0.3, 0.5]:
    result = process_temperature(temp, df.copy())
    final_results.append(result)

# Merge results back into the original DataFrame
for result in final_results:
    df = pd.concat([df, result], axis=1)


Iterating at temperature: 0.2


Temp 0.2: 100%|██████████| 357/357 [09:00<00:00,  1.51s/it] 


Iterating at temperature: 0.3


Temp 0.3: 100%|██████████| 357/357 [04:10<00:00,  1.43it/s]


Iterating at temperature: 0.5


Temp 0.5: 100%|██████████| 357/357 [05:10<00:00,  1.15it/s] 


In [19]:
df.to_csv('SemEval 2025 Test Data/final_datasets/pred_bulgarian_gold_label_data.csv', index=False)

In [20]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text,mode,output_temp_0.2,output_temp_0.3,output_temp_0.5
0,BG_670.txt,URW: Blaming the war on others rather than the...,URW: Blaming the war on others rather than the...,"Обвинява се управлението на Запада,че е претър...",Опитът на колективния Запад да „обезкърви Руси...,URW,"Доминиращата наратива ""Винене на войната в дру...","Доминантната наратива ""Винене на войната в дру...","Избраният доминиращ наратив ""Винене на войната..."
1,BG_3245.txt,"URW: Discrediting the West, Diplomacy","URW: Discrediting the West, Diplomacy: The Wes...",Коментира се евентуално прекратяване на подкре...,Подкрепата за Киев от страна на Запада вече не...,URW,"Избраният доминиращ наратив ""Дискредитиране на...","Избраният доминантен наратив ""Дискредитиране н...","В статията доминира разказът за ""Дискредитиран..."
2,A9_BG_5190.txt,"URW: Discrediting the West, Diplomacy",none,Неправителствената организация - Международнат...,"Дмитрий Медведев: НПО-та, спонсорирани от Соро...",URW,"Доминиращата наратива ""Дискредитиране на Запад...","Доминиращата наратива ""Дискредитиране на Запад...",None
3,A9_BG_3379.txt,"URW: Discrediting the West, Diplomacy",none,Санкциите и политиката на ЕС и САЩ след 2014-т...,Британски дипломат обвини Запада за украинския...,URW,"Доминиращата наратива ""Дискредитиране на Запад...","Основната наратива ""Дискредитиране на Запада, ...","Доминиращата наратива ""Дискредитиране на Запад..."
4,A7_URW_BG_3566.txt,URW: Discrediting Ukraine,URW: Discrediting Ukraine: Ukraine is a puppet...,В статията на няколко пъти САЩ са обвинени в т...,Ответните мерки ще бъдат крайно болезнени за Е...,URW,"Избраният доминиращ наратив ""Дискредитиране на...","Избраният доминиращ наратив ""Дискредитиране на...","Основната наративна линия на статията е ""Дискр..."


In [21]:
from huggingface_hub import login

login("")

from bert_score import score

for i in [0.2, 0.3, 0.5]:
    temp = round(i, 2)
    predictions = df[f'output_temp_{temp}'].tolist()
    references = df['ground_truth'].tolist()

    P, R, F1 = score(predictions, references, lang="en")

    df[f'precision_{temp}'] = P.tolist()
    df[f'recall_{temp}'] = R.tolist()
    df[f'f1_{temp}'] = F1.tolist()

/Users/rahulbouri/miniconda3/envs/sem-eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /Users/rahulbouri/.cache/huggingface/token
Login successful


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
df.head()

,article_id,dominant_narrative,sub_narratives,ground_truth,text,mode,output_temp_0.2,output_temp_0.3,output_temp_0.5,precision_0.2,recall_0.2,f1_0.2,precision_0.3,recall_0.3,f1_0.3,precision_0.5,recall_0.5,f1_0.5
0,BG_670.txt,URW: Blaming the war on others rather than the...,URW: Blaming the war on others rather than the...,"Обвинява се управлението на Запада,че е претър...",Опитът на колективния Запад да „обезкърви Руси...,URW,"Доминиращата наратива ""Винене на войната в дру...","Доминантната наратива ""Винене на войната в дру...","Избраният доминиращ наратив ""Винене на войната...",0.883232,0.917975,0.900268,0.882399,0.917816,0.899759,0.887834,0.920359,0.903804
1,BG_3245.txt,"URW: Discrediting the West, Diplomacy","URW: Discrediting the West, Diplomacy: The Wes...",Коментира се евентуално прекратяване на подкре...,Подкрепата за Киев от страна на Запада вече не...,URW,"Избраният доминиращ наратив ""Дискредитиране на...","Избраният доминантен наратив ""Дискредитиране н...","В статията доминира разказът за ""Дискредитиран...",0.886795,0.919972,0.903079,0.881494,0.912764,0.896856,0.882437,0.913815,0.897852
2,A9_BG_5190.txt,"URW: Discrediting the West, Diplomacy",none,Неправителствената организация - Международнат...,"Дмитрий Медведев: НПО-та, спонсорирани от Соро...",URW,"Доминиращата наратива ""Дискредитиране на Запад...","Доминиращата наратива ""Дискредитиране на Запад...",None,0.886331,0.905647,0.895885,0.889182,0.914018,0.901429,0.818014,0.621301,0.706215
3,A9_BG_3379.txt,"URW: Discrediting the West, Diplomacy",none,Санкциите и политиката на ЕС и САЩ след 2014-т...,Британски дипломат обвини Запада за украинския...,URW,"Доминиращата наратива ""Дискредитиране на Запад...","Основната наратива ""Дискредитиране на Запада, ...","Доминиращата наратива ""Дискредитиране на Запад...",0.875087,0.892236,0.883578,0.874206,0.893314,0.883657,0.874911,0.892878,0.883804
4,A7_URW_BG_3566.txt,URW: Discrediting Ukraine,URW: Discrediting Ukraine: Ukraine is a puppet...,В статията на няколко пъти САЩ са обвинени в т...,Ответните мерки ще бъдат крайно болезнени за Е...,URW,"Избраният доминиращ наратив ""Дискредитиране на...","Избраният доминиращ наратив ""Дискредитиране на...","Основната наративна линия на статията е ""Дискр...",0.907816,0.909300,0.908558,0.910814,0.913984,0.912396,0.908279,0.905951,0.907114


In [23]:
df.to_csv('SemEval 2025 Test Data/final_datasets/pred_bulgarian_gold_label_data.csv', index=False)

In [24]:
print(f"Mean Precision Temp 0.2: {df['precision_0.2'].mean()}")
print(f"Mean Recall Temp 0.2: {df['recall_0.2'].mean()}")
print(f"Mean F1 Temp 0.2: {df['f1_0.2'].mean()}")
print()
print(f"Mean Precision Temp 0.3: {df['precision_0.3'].mean()}")
print(f"Mean Recall Temp 0.3: {df['recall_0.3'].mean()}")
print(f"Mean F1 Temp 0.3: {df['f1_0.3'].mean()}")
print()
print(f"Mean Precision Temp 0.5: {df['precision_0.5'].mean()}")
print(f"Mean Recall Temp 0.5: {df['recall_0.5'].mean()}")
print(f"Mean F1 Temp 0.5: {df['f1_0.5'].mean()}")
print()

Mean Precision Temp 0.2: 0.8917610324731394
Mean Recall Temp 0.2: 0.9083801896966138
Mean F1 Temp 0.2: 0.8994993716060948

Mean Precision Temp 0.3: 0.8938636529345473
Mean Recall Temp 0.3: 0.9159767669456012
Mean F1 Temp 0.3: 0.9046631432047078

Mean Precision Temp 0.5: 0.893611924154084
Mean Recall Temp 0.5: 0.9166004767938822
Mean F1 Temp 0.5: 0.9048831162332487

